# NeuroFusionAI: Step 2 - Model Training & Evaluation
This notebook trains and evaluates models for each individual modality, and then trains the multimodal fusion network.
Specifically:
1. **Image Classifier**: Fine-tunes ResNet-18 on drawings.
2. **Voice Classifier**: Trains an MLP (PyTorch) and XGBoost Classifier on voice features.
3. **Telemonitoring Regressor**: Trains an MLP (PyTorch) and XGBoost Regressor on UPDRS scores.
4. **Multimodal Fusion**: Trains a late/mid-fusion network using combined embeddings.


In [ ]:
import os
import sys
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, r2_score, mean_squared_error

# Resolve project root (walk up until we find the 'datasets' directory)
PROJECT_ROOT = os.path.abspath('.')
for p in [os.path.abspath(x) for x in sys.path if x]:
    if os.path.isdir(os.path.join(p, 'datasets')):
        PROJECT_ROOT = p
        break
    elif os.path.isdir(os.path.join(os.path.dirname(p), 'datasets')):
        PROJECT_ROOT = os.path.dirname(p)
        break
while not os.path.isdir(os.path.join(PROJECT_ROOT, 'datasets')) and PROJECT_ROOT != os.path.dirname(PROJECT_ROOT):
    PROJECT_ROOT = os.path.dirname(PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

# Helper to resolve paths relative to project root
def P(*parts):
    return os.path.join(PROJECT_ROOT, *parts)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
os.makedirs(P('outputs', 'checkpoints'), exist_ok=True)
print('Project root:', PROJECT_ROOT, '| Device:', device)


## 1. Train Drawings Classifier (Image Model)
We load the drawing images, instantiate the `ImageDrawingClassifier` model, and train for 2 epochs.


In [ ]:
from preprocessing.image_preprocessing import get_image_dataloader
from models.image_model import ImageDrawingClassifier

train_loader = get_image_dataloader('train', batch_size=16, shuffle=True, augment=True)
val_loader = get_image_dataloader('validation', batch_size=16, shuffle=False)

model = ImageDrawingClassifier(num_classes=2, pretrained=True).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.resnet.fc.parameters(), lr=0.001)

# Training loop (2 epochs for fast verification)
for epoch in range(2):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * labels.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)
    
    # Val eval
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            _, preds = outputs.max(1)
            val_correct += preds.eq(labels).sum().item()
            val_total += labels.size(0)
            
    print(f'Epoch {epoch+1}/2 | Loss: {running_loss/total:.4f} | Train Acc: {correct/total:.4f} | Val Acc: {val_correct/val_total:.4f}')

# Save model checkpoint
torch.save(model.state_dict(), P('outputs', 'checkpoints', 'image_best.pth'))
print('Image Classifier saved!')


## 1.1 Train MRI Classifier (Neuroimaging Model)
We fine-tune the pretrained EfficientNet-B0 model on the preprocessed MRI images.


In [ ]:
from preprocessing.mri_preprocessing import get_mri_dataloader
from models.mri_model import MRIClassifier

mri_train_loader = get_mri_dataloader('train', batch_size=16, shuffle=True, augment=True)
mri_val_loader = get_mri_dataloader('validation', batch_size=16, shuffle=False)

mri_model = MRIClassifier(num_classes=2, pretrained=True).to(device)
mri_criterion = nn.CrossEntropyLoss()
mri_optimizer = optim.Adam(mri_model.efficientnet.classifier.parameters(), lr=0.001)

# Training loop (2 epochs for fast verification)
for epoch in range(2):
    mri_model.train()
    running_loss, correct, total = 0.0, 0, 0
    for imgs, labels in mri_train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        mri_optimizer.zero_grad()
        outputs = mri_model(imgs)
        loss = mri_criterion(outputs, labels)
        loss.backward()
        mri_optimizer.step()
        
        running_loss += loss.item() * labels.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)
    
    # Val eval
    mri_model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for imgs, labels in mri_val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = mri_model(imgs)
            _, preds = outputs.max(1)
            val_correct += preds.eq(labels).sum().item()
            val_total += labels.size(0)
            
    print(f'Epoch {epoch+1}/2 | Loss: {running_loss/total:.4f} | Train Acc: {correct/total:.4f} | Val Acc: {val_correct/val_total:.4f}')

# Save model checkpoint
torch.save(mri_model.state_dict(), P('outputs', 'checkpoints', 'mri_best.pth'))
print('MRI Classifier saved!')


## 2. Train Voice Classifier (Acoustics Model)
We evaluate both PyTorch MLP and XGBoost models on the Oxford voice dataset.


In [ ]:
from preprocessing.voice_preprocessing import get_voice_dataloader
from models.voice_model import VoiceMLPClassifier, VoiceXGBClassifier

voice_train_loader = get_voice_dataloader('train', batch_size=16, shuffle=True)
voice_val_loader = get_voice_dataloader('validation', batch_size=16, shuffle=False)

# A. PyTorch MLP Classifier
voice_model = VoiceMLPClassifier(input_dim=22, hidden_dim=64, num_classes=2).to(device)
voice_criterion = nn.CrossEntropyLoss()
# Added weight decay (L2 regularization) to improve generalization on the voice features
voice_opt = optim.Adam(voice_model.parameters(), lr=0.005, weight_decay=1e-4)

best_val_acc = 0.0
best_model_state = None

print('Training Voice MLP Classifier...')
for epoch in range(10):
    voice_model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    for feats, labels in voice_train_loader:
        feats, labels = feats.to(device), labels.to(device)
        voice_opt.zero_grad()
        outputs = voice_model(feats)
        loss = voice_criterion(outputs, labels)
        loss.backward()
        voice_opt.step()
        
        train_loss += loss.item() * labels.size(0)
        _, preds = outputs.max(1)
        train_correct += preds.eq(labels).sum().item()
        train_total += labels.size(0)
    
    # Epoch validation
    voice_model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    val_preds, val_targets = [], []
    with torch.no_grad():
        for feats, labels in voice_val_loader:
            feats, labels = feats.to(device), labels.to(device)
            outputs = voice_model(feats)
            loss = voice_criterion(outputs, labels)
            val_loss += loss.item() * labels.size(0)
            _, preds = outputs.max(1)
            val_correct += preds.eq(labels).sum().item()
            val_total += labels.size(0)
            val_preds.extend(preds.cpu().numpy())
            val_targets.extend(labels.cpu().numpy())
            
    val_acc = accuracy_score(val_targets, val_preds)
    val_f1 = f1_score(val_targets, val_preds)
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_state = {k: v.cpu().clone() for k, v in voice_model.state_dict().items()}
        
    print(f'Epoch {epoch+1:02d}/10 | Train Loss: {train_loss/train_total:.4f} | Train Acc: {train_correct/train_total:.4f} | Val Loss: {val_loss/val_total:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}')

# Restore best validation model weights
if best_model_state is not None:
    voice_model.load_state_dict(best_model_state)

voice_model.eval()
val_preds, val_targets = [], []
with torch.no_grad():
    for feats, labels in voice_val_loader:
        feats = feats.to(device)
        outputs = voice_model(feats)
        _, preds = outputs.max(1)
        val_preds.extend(preds.cpu().numpy())
        val_targets.extend(labels.numpy())

mlp_acc = accuracy_score(val_targets, val_preds)
mlp_precision = precision_score(val_targets, val_preds, zero_division=0)
mlp_recall = recall_score(val_targets, val_preds, zero_division=0)
mlp_f1 = f1_score(val_targets, val_preds, zero_division=0)

print('\n--- Voice MLP Best Model Results ---')
print(f'Validation Accuracy : {mlp_acc:.4f}')
print(f'Validation Precision: {mlp_precision:.4f}')
print(f'Validation Recall   : {mlp_recall:.4f}')
print(f'Validation F1-Score : {mlp_f1:.4f}')
torch.save(voice_model.state_dict(), P('outputs', 'checkpoints', 'voice_mlp_best.pth'))
print('Saved Best Voice MLP Classifier weights!')

# B. XGBoost Classifier
print('\nTraining Voice XGBoost Classifier...')
train_df = pd.read_csv(P('datasets', 'train', 'voice', 'oxford_train.csv'))
val_df = pd.read_csv(P('datasets', 'validation', 'voice', 'oxford_validation.csv'))

X_train = train_df.drop(columns=['status']).values
y_train = train_df['status'].values
X_val = val_df.drop(columns=['status']).values
y_val = val_df['status'].values

# Optimized hyperparameters: n_estimators tuned to 100, max_depth to 4, learning_rate to 0.05 for regularization
xgb_model = VoiceXGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.05)
xgb_model.fit(X_train, y_train)
xgb_preds = xgb_model.predict(X_val)

xgb_acc = accuracy_score(y_val, xgb_preds)
xgb_precision = precision_score(y_val, xgb_preds, zero_division=0)
xgb_recall = recall_score(y_val, xgb_preds, zero_division=0)
xgb_f1 = f1_score(y_val, xgb_preds, zero_division=0)

print('--- Voice XGBoost Results ---')
print(f'Validation Accuracy : {xgb_acc:.4f}')
print(f'Validation Precision: {xgb_precision:.4f}')
print(f'Validation Recall   : {xgb_recall:.4f}')
print(f'Validation F1-Score : {xgb_f1:.4f}')
xgb_model.save(P('outputs', 'checkpoints', 'voice_xgb_best.pkl'))
print('Saved Voice XGBoost Classifier model!')


## 3. Train Telemonitoring Regressor (UPDRS Severity Model)
We train a model to predict the motor and total UPDRS scores from telemonitoring.


In [ ]:
from preprocessing.telemonitor_preprocessing import get_telemonitor_dataloader
from models.telemonitor_model import TelemonitorMLPRegressor, TelemonitorXGBRegressor

tele_train_loader = get_telemonitor_dataloader('train', batch_size=32, shuffle=True)
tele_val_loader = get_telemonitor_dataloader('validation', batch_size=32, shuffle=False)

# A. PyTorch MLP Regressor
tele_model = TelemonitorMLPRegressor(input_dim=19, hidden_dim=64, output_dim=2).to(device)
tele_criterion = nn.MSELoss()
# Added weight decay for regularization to prevent overfitting on the telemonitoring data
tele_opt = optim.Adam(tele_model.parameters(), lr=0.005, weight_decay=1e-4)

best_val_r2 = -float('inf')
best_tele_state = None

print('Training Telemonitoring MLP Regressor...')
for epoch in range(10):
    tele_model.train()
    train_loss, train_total = 0.0, 0
    for feats, targets in tele_train_loader:
        feats, targets = feats.to(device), targets.to(device)
        tele_opt.zero_grad()
        preds = tele_model(feats)
        loss = tele_criterion(preds, targets)
        loss.backward()
        tele_opt.step()
        
        train_loss += loss.item() * targets.size(0)
        train_total += targets.size(0)
        
    # Epoch validation
    tele_model.eval()
    val_loss = 0.0
    val_preds, val_targets = [], []
    with torch.no_grad():
        for feats, targets in tele_val_loader:
            feats, targets = feats.to(device), targets.to(device)
            preds = tele_model(feats)
            loss = tele_criterion(preds, targets)
            val_loss += loss.item() * targets.size(0)
            val_preds.append(preds.cpu().numpy())
            val_targets.append(targets.cpu().numpy())
            
    val_preds = np.concatenate(val_preds, axis=0)
    val_targets = np.concatenate(val_targets, axis=0)
    val_r2 = r2_score(val_targets, val_preds)
    val_mse = mean_squared_error(val_targets, val_preds)
    
    if val_r2 > best_val_r2:
        best_val_r2 = val_r2
        best_tele_state = {k: v.cpu().clone() for k, v in tele_model.state_dict().items()}
        
    print(f'Epoch {epoch+1:02d}/10 | Train Loss (MSE): {train_loss/train_total:.4f} | Val Loss (MSE): {val_mse:.4f} | Val R^2: {val_r2:.4f}')

# Restore best validation model weights
if best_tele_state is not None:
    tele_model.load_state_dict(best_tele_state)

tele_model.eval()
val_preds, val_targets = [], []
with torch.no_grad():
    for feats, targets in tele_val_loader:
        feats = feats.to(device)
        preds = tele_model(feats)
        val_preds.append(preds.cpu().numpy())
        val_targets.append(targets.numpy())

val_preds = np.concatenate(val_preds, axis=0)
val_targets = np.concatenate(val_targets, axis=0)
mlp_r2 = r2_score(val_targets, val_preds)
mlp_mse = mean_squared_error(val_targets, val_preds)

print('\n--- Telemonitor MLP Best Model Results ---')
print(f'Validation MSE     : {mlp_mse:.4f}')
print(f'Validation R^2 Score: {mlp_r2:.4f}')
torch.save(tele_model.state_dict(), P('outputs', 'checkpoints', 'telemonitor_mlp_best.pth'))
print('Saved Best Telemonitor MLP Regressor weights!')

# B. XGBoost Regressor
print('\nTraining Telemonitoring XGBoost Regressor...')
train_tele_df = pd.read_csv(P('datasets', 'train', 'telemonitoring', 'telemonitor_train.csv'))
val_tele_df = pd.read_csv(P('datasets', 'validation', 'telemonitoring', 'telemonitor_validation.csv'))

X_train_t = train_tele_df.drop(columns=['motor_UPDRS', 'total_UPDRS']).values
y_train_t = train_tele_df[['motor_UPDRS', 'total_UPDRS']].values
X_val_t = val_tele_df.drop(columns=['motor_UPDRS', 'total_UPDRS']).values
y_val_t = val_tele_df[['motor_UPDRS', 'total_UPDRS']].values

# Optimized hyperparameters: n_estimators tuned to 100, max_depth to 4, learning_rate to 0.05
xgb_reg = TelemonitorXGBRegressor(n_estimators=100, max_depth=4, learning_rate=0.05)
xgb_reg.fit(X_train_t, y_train_t)
xgb_preds_t = xgb_reg.predict(X_val_t)

xgb_r2 = r2_score(y_val_t, xgb_preds_t)
xgb_mse = mean_squared_error(y_val_t, xgb_preds_t)

print('--- Telemonitor XGBoost Results ---')
print(f'Validation MSE     : {xgb_mse:.4f}')
print(f'Validation R^2 Score: {xgb_r2:.4f}')
xgb_reg.save(P('outputs', 'checkpoints', 'telemonitor_xgb_best.pkl'))
print('Saved Telemonitor XGBoost Regressor model!')


## 4. Train Multimodal Fusion Network
We train the joint classifier using the projection head design on fused representations.


In [ ]:
from preprocessing.fusion_preprocessing import get_fusion_dataloader
from models.fusion_model import MultimodalClassifier

fusion_train_loader = get_fusion_dataloader('train', batch_size=16)
fusion_val_loader = get_fusion_dataloader('validation', batch_size=16)

fusion_model = MultimodalClassifier(image_dim=256, mri_dim=256, voice_dim=22, clinical_dim=19, fusion_dim=32, num_classes=2).to(device)
fusion_criterion = nn.CrossEntropyLoss()
fusion_opt = optim.Adam(fusion_model.parameters(), lr=0.005)

for epoch in range(5):
    fusion_model.train()
    correct, total, running_loss = 0, 0, 0.0
    for img_emb, mri_emb, voice_feat, clinical_feat, label in fusion_train_loader:
        img_emb = img_emb.to(device)
        mri_emb = mri_emb.to(device)
        voice_feat = voice_feat.to(device)
        clinical_feat = clinical_feat.to(device)
        label = label.to(device)
        
        fusion_opt.zero_grad()
        outputs = fusion_model(img_emb, mri_emb, voice_feat, clinical_feat)
        loss = fusion_criterion(outputs, label)
        loss.backward()
        fusion_opt.step()
        
        running_loss += loss.item() * label.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(label).sum().item()
        total += label.size(0)
        
    # Eval
    fusion_model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for img_emb, mri_emb, voice_feat, clinical_feat, label in fusion_val_loader:
            img_emb = img_emb.to(device)
            mri_emb = mri_emb.to(device)
            voice_feat = voice_feat.to(device)
            clinical_feat = clinical_feat.to(device)
            label = label.to(device)
            
            outputs = fusion_model(img_emb, mri_emb, voice_feat, clinical_feat)
            _, preds = outputs.max(1)
            val_correct += preds.eq(label).sum().item()
            val_total += label.size(0)
            
    print(f'Epoch {epoch+1}/5 | Train Acc: {correct/total:.4f} | Val Acc: {val_correct/val_total:.4f}')

torch.save(fusion_model.state_dict(), P('outputs', 'checkpoints', 'fusion_best.pth'))
print('Multimodal Fusion model saved successfully!')
